# Module 4: RAG Knowledge Base — Build FAISS Index (LOCAL)
Dataset pulled via HuggingFace `datasets` API.
Embeddings: sentence-transformers/all-MiniLM-L6-v2 (tiny, runs fine on your GPU
or even CPU -- ~80MB model).
Vector store: FAISS flat index (exact search, dataset is small enough this beats
approximate search on simplicity).

In [1]:
# pip install sentence-transformers faiss-cpu datasets
# (faiss-cpu is intentional even though you have a GPU -- FAISS GPU builds are
# fiddly to install and totally unnecessary at this dataset size; MiniLM encoding
# is the only part that benefits from GPU, and sentence-transformers will use
# your GPU automatically if available)

import pandas as pd
import numpy as np
import json
import os
import faiss
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

print("CUDA available for embedding model:", torch.cuda.is_available())

CUDA available for embedding model: False


## Load dataset

In [2]:
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()
print(df.shape)

(26872, 5)


## Clean + dedupe

In [3]:
df = df.dropna(subset=["instruction", "response"]).reset_index(drop=True)
df["instruction"] = df["instruction"].astype(str).str.strip()
df["response"] = df["response"].astype(str).str.strip()

before = len(df)
df = df.drop_duplicates(subset=["instruction"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicates, {len(df)} remain")

Dropped 2237 duplicates, 24635 remain


## Load embedding model (auto-uses GPU if available, else CPU)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print("Embedding dim:", embed_model.get_sentence_embedding_dimension(), "| device:", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim: 384 | device: cpu


/tmp/ipykernel_63344/454089514.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dim:", embed_model.get_sentence_embedding_dimension(), "| device:", device)


## Embed all instructions

In [5]:
instructions = df["instruction"].tolist()

embeddings = embed_model.encode(
    instructions, batch_size=128, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)
print(embeddings.shape)

Batches:   0%|          | 0/193 [00:00<?, ?it/s]

(24635, 384)


## Build FAISS index

In [6]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype("float32"))
print("Index size:", index.ntotal)

Index size: 24635


## Sanity check

In [7]:
def search(query, k=3):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, k)
    for score, idx in zip(scores[0], idxs[0]):
        print(f"score={score:.3f} | Q: {df.iloc[idx]['instruction']}")
        print(f"  A: {df.iloc[idx]['response'][:150]}...")

search("where is my package, it's late")

score=0.738 | Q: where can I see when my damn package is going to arrive?
  A: I understand your frustration and eagerness to track your package. To provide you with the most accurate information about your delivery, could you pl...
score=0.736 | Q: where do I see when my package is going to arrive?
  A: We understand your eagerness to track the arrival of your package. To provide you with the most accurate information, could you please provide us with...
score=0.726 | Q: where can I see when my package is going to arrive?
  A: We understand your eagerness to track the progress and estimated arrival time of your package. To check the delivery status, you can visit our website...


## Save index + chunk metadata directly into local_app/models/rag/

In [8]:
OUTPUT_DIR = "../local_app/models/rag"
os.makedirs(OUTPUT_DIR, exist_ok=True)

faiss.write_index(index, os.path.join(OUTPUT_DIR, "faiss_index.bin"))

chunks = df[["instruction", "response", "intent", "category"]].to_dict(orient="records")
with open(os.path.join(OUTPUT_DIR, "chunks.json"), "w") as f:
    json.dump(chunks, f, indent=2)

print("Saved to", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))
print(f"Index has {index.ntotal} vectors, chunks.json has {len(chunks)} entries")

Saved to ../local_app/models/rag
['.gitkeep', 'faiss_index.bin', 'chunks.json']
Index has 24635 vectors, chunks.json has 24635 entries
